# Pipeline conversacional de Chatbot com NLP (chatbotv3)

- **Repositório:** `nlp-llms-portfolio`  
- **Tecnologias:** NLTK, spaCy, Scikit-Learn  
- **Arquivo Original:** `ChatBotV3.ipynb`  

---


In [3]:
# Crie um ambiente virtual Python
!python -m venv /tmp/colab_venv
!source /tmp/colab_venv/bin/activate && \
 pip install --no-cache-dir numpy==1.23.5 && \
 pip install --no-cache-dir spacy==3.6.1 && \
 python -m spacy download pt_core_news_lg

/bin/bash: line 1: python: command not found
/bin/bash: line 1: /tmp/colab_venv/bin/activate: No such file or directory


In [4]:
import sys
sys.path.insert(0, '/tmp/colab_venv/lib/python3.10/site-packages')

In [5]:
try:
    import numpy as np
    import spacy

    print(f"✅ NumPy {np.__version__} carregado com sucesso!")
    print(f"✅ spaCy {spacy.__version__} carregado com sucesso!")

    nlp = spacy.load("pt_core_news_lg")
    print("✅ Modelo de linguagem carregado!")
except Exception as e:
    print(f"❌ Erro: {str(e)}")
    print("\nExecute este comando como último recurso:")
    print("!pip install --force-reinstall numpy==1.21.6 spacy==3.2.0")

❌ Erro: No module named 'spacy'

Execute este comando como último recurso:
!pip install --force-reinstall numpy==1.21.6 spacy==3.2.0


## Instalação e Importação Bibliotecas



In [1]:
# Importação de módulos
import re, string
import numpy as np
import requests
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import RSLPStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import spacy
from gensim.models import Word2Vec
import multiprocessing

# Baixar recursos NLTK para Português
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('rslp')

# Carregar modelo spaCy para português
nlp = spacy.load("pt_core_news_lg")


AttributeError: module 'numpy' has no attribute 'dtypes'

## Carregar e Segmentar o Texto

In [ ]:
url = 'https://raw.githubusercontent.com/VictorSilvaCamargo/o-guarani/main/o-guarani.txt'
resposta = requests.get(url)
if resposta.status_code == 200:
    livro = resposta.text
    print(f"Livro carregado! Total de caracteres: {len(livro)}")
else:
    print("Erro ao carregar o livro:", resposta.status_code)

# Dividir em sentenças (útil para criar chunks de contexto)
sentencas_originais = sent_tokenize(livro, language='portuguese')
print(f"Total de sentenças: {len(sentencas_originais)}")


## Pré-processamento geral do texto

In [ ]:
def preprocessar_texto(texto):
    texto = re.sub(r'(?<!\w)[\.,;:!\?](?!\w)', '', texto)  # remover pontuação específica
    texto = texto.lower()
    texto = re.sub(r'\s+', ' ', texto).strip()
    doc = nlp(texto)
    tokens_filtrados = [
        token.lemma_
        for token in doc
        if not token.is_stop and token.is_alpha and len(token.text) > 2
    ]
    return tokens_filtrados

tokens_processados = preprocessar_texto(livro)
print(f"Total de tokens (lematizados) processados: {len(tokens_processados)}")


## Pré-processamento para Word2Vec

In [ ]:
def preprocessar_texto_para_word2vec(texto):
    sentencas = sent_tokenize(texto, language='portuguese')
    sentencas_processadas = []
    for sent in sentencas:
        sent = re.sub(r'(?<!\w)[\.,;:!\?](?!\w)', '', sent)
        sent = sent.lower().strip()
        doc = nlp(sent)
        tokens_filtrados = [
            token.lemma_
            for token in doc
            if not token.is_stop and token.is_alpha and len(token.text) > 2
        ]
        if len(tokens_filtrados) > 3:
            sentencas_processadas.append(tokens_filtrados)
    return sentencas_processadas

sentencas_processadas = preprocessar_texto_para_word2vec(livro)
print(f"Total de sentenças processadas: {len(sentencas_processadas)}")


## Treinamento de embeddings Word2Vec

In [ ]:
model_w2v = Word2Vec(
    sentences=sentencas_processadas,
    vector_size=100,
    window=5,
    sg=1,             # skip-gram
    min_count=3,      # ignora palavras raras
    workers=multiprocessing.cpu_count(),
    epochs=15
)
model_w2v.save("word2vec_skipgram_o_guarani.model")
print("Word2Vec treinado e salvo.")


## Entidades nomeadas e processamento avançado


In [ ]:
def normalizar_entidade(ent):
    ent = ent.lower().strip()
    ent = re.sub(r"[^\w\s]", "", ent)
    ent = re.sub(r"\s+", " ", ent)
    return ent

def extrair_entidades_filtradas(texto):
    doc = nlp(texto)
    entidades = {"PER": set(), "LOC": set()}
    for ent in doc.ents:
        if ent.label_ in entidades:
            ent_text = normalizar_entidade(ent.text)
            if ent.label_ == "PER" and len(ent_text.split()) > 1:
                entidades["PER"].add(ent_text)
            elif ent.label_ == "LOC":
                entidades["LOC"].add(ent_text)
    return entidades


## Criação de chunks

In [ ]:
# Preparar stemmer e stopwords
stemmer_global = RSLPStemmer()
stop_words_global = set(stopwords.words("portuguese"))

def preprocessar_sentenca(sent):
    sent = sent.lower()
    sent = re.sub(r"\s+", " ", sent).strip()
    sent = re.sub(r"[^\w\s]", "", sent)
    tokens = word_tokenize(sent, language="portuguese")
    return [
        stemmer_global.stem(w)
        for w in tokens
        if (w not in stop_words_global and w.isalpha() and len(w) > 2)
    ]

def criar_chunks(sentencas, tamanho_tokens=100, sobreposicao=0.3):
    chunks = []
    i = 0
    passo = int(tamanho_tokens * (1 - sobreposicao))
    while i < len(sentencas):
        tokens_acumulados = []
        texto_acumulado = []
        j = i
        while j < len(sentencas) and len(tokens_acumulados) < tamanho_tokens:
            texto_acumulado.append(sentencas[j])
            ts = preprocessar_sentenca(sentencas[j])
            tokens_acumulados.extend(ts)
            j += 1
        if len(tokens_acumulados) >= tamanho_tokens:
            chunk_texto = " ".join(texto_acumulado)
            chunks.append((tokens_acumulados[:tamanho_tokens], chunk_texto))
        i += passo
    return chunks

chunks_legiveis = criar_chunks(sentencas_originais, tamanho_tokens=100, sobreposicao=0.3)
print(f"Total de chunks criados: {len(chunks_legiveis)}")


## Vetorização dos chunks com TF-IDF + Word2Vec

In [ ]:
# Função que calcula vetor médio ponderado de tokens de um chunk
def vetor_ponderado_chunk(tokens, modelo, tfidf_vectorizer):
    vetores = []
    pesos = []
    for token in tokens:
        if token in modelo.wv:
            idx = tfidf_vectorizer.vocabulary_.get(token)
            peso = tfidf_vectorizer.idf_[idx] if idx is not None else 1.0
            vetores.append(modelo.wv[token])
            pesos.append(peso)
    if vetores:
        return np.average(vetores, axis=0, weights=pesos)
    else:
        return np.zeros(modelo.vector_size)

# Treinar TF-IDF no corpus inteiro (lista de tokens do livro)
tfidf_vectorizer = TfidfVectorizer()
tfidf_vectorizer.fit(tokens_processados)  # usa os tokens de preprocessamento geral

# Mapear cada chunk para (vetor, texto, entidades)
mapa_chunks = []
for tokens_chunk, texto_chunk in chunks_legiveis:
    vetor = vetor_ponderado_chunk(tokens_chunk, model_w2v, tfidf_vectorizer)
    entidades_chunk = extrair_entidades_filtradas(texto_chunk)
    mapa_chunks.append((vetor, texto_chunk, entidades_chunk))

print("Vetorização dos chunks concluída.")


## Sistema de Perguntas e Respostas

In [ ]:
def vetorizar_pergunta(pergunta, modelo):
    pergunta = pergunta.lower().strip()
    pergunta = re.sub(r"[^\w\s]", "", pergunta)
    doc = nlp(pergunta)
    tokens_filtrados = [
        token.lemma_ for token in doc
        if not token.is_stop and token.is_alpha and len(token.text) > 2
    ]
    vetores = [modelo.wv[w] for w in tokens_filtrados if w in modelo.wv]
    if vetores:
        return np.mean(vetores, axis=0).reshape(1, -1)
    else:
        return None

def extrair_entidades_pergunta(pergunta):
    doc = nlp(pergunta)
    resultado = set()
    for ent in doc.ents:
        if ent.label_ == "PER":
            ent_norm = normalizar_entidade(ent.text)
            if len(ent_norm.split()) >= 2:
                resultado.add(ent_norm)
    return resultado

def responder_pergunta(pergunta, modelo, mapa_chunks, topn=3):
    vet_pergunta = vetorizar_pergunta(pergunta, modelo)
    if vet_pergunta is None:
        print("Não foi possível vetorizar a pergunta (palavras não reconhecidas).")
        return
    # Calcula similaridade de cosseno entre pergunta e cada chunk
    vetores_chunks = np.array([c[0] for c in mapa_chunks])
    similaridades = cosine_similarity(vet_pergunta, vetores_chunks)[0]
    # Seleciona candidatos (maiores similaridades)
    candidatos = similaridades.argsort()[-(topn*3):][::-1]

    entidades_perg = extrair_entidades_pergunta(pergunta)
    pontuacoes_ajustadas = []
    for idx in candidatos:
        vetor_chunk, texto_chunk, entidades_chunk = mapa_chunks[idx]
        per_chunk = entidades_chunk["PER"]
        qtd_match_per = len(entidades_perg.intersection(per_chunk))
        bonus_ent = 0.2 * qtd_match_per
        score_ajustado = similaridades[idx] + bonus_ent
        pontuacoes_ajustadas.append((idx, score_ajustado))
    pontuacoes_ajustadas.sort(key=lambda x: x[1], reverse=True)
    selecionados = pontuacoes_ajustadas[:topn]

    print(f"\nTop {topn} resposta(s):")
    for idx, score in selecionados:
        _, texto_chunk, entidades_chunk = mapa_chunks[idx]
        print(f"\n🔹 Score ajustado: {score:.4f}")
        print(texto_chunk.strip())
        pess = ", ".join(entidades_chunk["PER"]) or "—"
        locs = ", ".join(entidades_chunk["LOC"]) or "—"
        print(f"   → Pessoas mencionadas: {pess}")
        print(f"   → Locais mencionados:  {locs}")


In [ ]:
# Exemplo de pergunta
pergunta_usuario = "Como Peri salvou Cecília?"
responder_pergunta(pergunta_usuario, model_w2v, mapa_chunks, topn=1)
